In [2]:
import pandas as pd
df=pd.read_csv("datasets/annotated_genomes_ncbi_5_27_26.tsv",sep="\t")


In [1]:
from ete3 import Tree, NCBITaxa
import re
import sys


HEXAPODA_TAXID = 6960  # NCBI taxid for Insecta





def resolve_taxid(ncbi, species_name):
    """
    Resolve a tree leaf label to an NCBI taxid.
    Returns None if the label cannot be resolved.
    """
    name2taxid = ncbi.get_name_translator([species_name])

    if species_name not in name2taxid:
        return None

    return name2taxid[species_name][0]


def get_lineage(ncbi, taxid):
    """
    Safely return the NCBI lineage for a taxid.
    """
    try:
        return ncbi.get_lineage(taxid)
    except Exception:
        return None


def is_insect_taxid(ncbi, taxid):
    """
    Return True if a taxid belongs to Insecta.
    """
    lineage = get_lineage(ncbi, taxid)

    if lineage is None:
        return False

    return HEXAPODA_TAXID in lineage


def get_rank_name_from_taxid(ncbi, taxid, target_rank):
    """
    Return the taxon name at a requested NCBI rank for a taxid.
    For example, target_rank='order'.
    """
    lineage = get_lineage(ncbi, taxid)

    if lineage is None:
        return None

    ranks = ncbi.get_rank(lineage)
    target_taxids = [tid for tid in lineage if ranks.get(tid) == target_rank]

    if not target_taxids:
        return None

    names = ncbi.get_taxid_translator(target_taxids)
    return names[target_taxids[0]]


def prune_non_hexapoda_leaves(tree, ncbi):
    """
    Delete all non-insect leaves from the tree.

    Unresolved leaves are also deleted, because they cannot be confirmed
    as insects.
    """
    leaves_to_keep = []

    for leaf in tree.iter_leaves():
        old_name = leaf.name
        taxid = resolve_taxid(ncbi, old_name)

        if taxid is None:
            sys.stderr.write(
                f"WARNING: Could not resolve {old_name}; deleting leaf\n"
            )
            continue

        if not is_insect_taxid(ncbi, taxid):
            sys.stderr.write(
                f"Deleting non-hexapod leaf: {old_name}\n"
            )
            continue

        leaves_to_keep.append(leaf)

    tree.prune(leaves_to_keep, preserve_branch_length=True)


def relabel_leaves_by_order(tree, ncbi):
    """
    Rename each remaining hexapod leaf to its order name.

    Assumes the tree has already been pruned to insects only.
    """
    for leaf in tree.iter_leaves():
        old_name = leaf.name
        taxid = resolve_taxid(ncbi, old_name)

        if taxid is None:
            try:
                taxid=resolve_taxid(ncbi, old_name.strip().split(" ")[0])
            except:
                    
                sys.stderr.write(
                    f"WARNING: Could not resolve {old_name}; keeping original label\n"
                )
                leaf.order=old_name
                continue
       

        order_name = get_rank_name_from_taxid(ncbi, taxid, "order")

        if order_name is None:
            sys.stderr.write(
                f"WARNING: Could not find order for {old_name}; keeping original label\n"
            )
            leaf.order=old_name
            continue
 

        leaf.order = order_name


def collapse_monophyletic_terminal_groups(tree):
    """
    Collapse monophyletic terminal groups by leaf.order using ETE3's
    get_monophyletic().

    For each order, this keeps one representative leaf per monophyletic
    occurrence of that order. If the same order is non-monophyletic and appears
    in multiple parts of the tree, one representative is kept from each part.

    Because this uses tree.prune(), the resulting tree keeps the original
    root-to-tip structure for retained leaves, which helps preserve ultrametricity.

    Assumes each leaf has:
        leaf.order
    """
    representatives = []

    orders = sorted({leaf.order for leaf in tree.iter_leaves()})

    for order in orders:
        monophyletic_nodes = tree.get_monophyletic(
            values=[order],
            target_attr="order"
        )

        for node in monophyletic_nodes:
            representative_leaf = node.get_leaves()[0]
            representatives.append(representative_leaf)

    # Remove duplicates, in case a leaf is returned more than once
    seen = set()
    unique_representatives = []

    for leaf in representatives:
        if leaf not in seen:
            unique_representatives.append(leaf)
            seen.add(leaf)

    tree.prune(unique_representatives, preserve_branch_length=True)

    for leaf in tree.iter_leaves():
        leaf.name = leaf.order



/n/home11/rkapoor/.conda/envs/rishabh/lib/python3.9/site-packages/scipy/__init__.py:146: UserWarning: A NumPy version >=1.16.5 and <1.23.0 is required for this version of SciPy (detected version 1.23.5
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"


In [3]:
tree = Tree("datasets/phylogenetic_trees/timetree_Misof_2014.nwk")

ncbi = NCBITaxa()
prune_non_hexapoda_leaves(tree, ncbi)
relabel_leaves_by_order(tree, ncbi)
collapse_monophyletic_terminal_groups(tree)



collapse_monophyletic_terminal_groups(tree)
tree.prune(list(set(df[df['class'].isin(['Collembola','Insecta'])].order)),preserve_branch_length=True)
print(tree)

Deleting non-hexapod leaf: Symphylella vulgaris
Deleting non-hexapod leaf: Glomeris pustulata
Deleting non-hexapod leaf: Leptuca pugilator
Deleting non-hexapod leaf: Penaeus vannamei
Deleting non-hexapod leaf: Sarsinebalia urgorrii
Deleting non-hexapod leaf: Cypridinidae
Deleting non-hexapod leaf: Lepeophtheirus salmonis
Deleting non-hexapod leaf: Daphnia pulex
Deleting non-hexapod leaf: Xibalbanus tulumensis
Deleting non-hexapod leaf: Ixodes scapularis



                              /-Lepidoptera
                           /-|
                          |   \-Trichoptera
                        /-|
                       |  |   /-Diptera
                       |   \-|
                     /-|      \-Siphonaptera
                    |  |
                    |  |   /-Coleoptera
                  /-|   \-|
                 |  |      \-Neuroptera
               /-|  |
              |  |   \-Hymenoptera
              |  |
            /-|   \-Psocodea
           |  |
           |  |   /-Hemiptera
           |   \-|
         /-|      \-Thysanoptera
        |  |
        |  |      /-Blattodea
        |  |   /-|
      /-|   \-|   \-Phasmatodea
     |  |     |
     |  |      \-Orthoptera
     |  |
   /-|  |   /-Ephemeroptera
  |  |   \-|
  |  |      \-Odonata
--|  |
  |   \-Zygentoma
  |
  |   /-Symphypleona
   \-|
      \-Entomobryomorpha


In [4]:

tree.write(
    outfile="datasets/phylogenetic_trees/timetree_Misof_2014_insect_orders_collapsed_pruned.nwk",
    format=1
)


## Bernot pancrustacea

In [177]:
bernot = Tree("datasets/phylogenetic_trees/Bernot_2023.newick")

In [178]:
relabel_leaves_by_order(bernot, ncbi)
collapse_monophyletic_terminal_groups(bernot)


In [179]:
HEXAPODA_TAXID = 6960
CHELICERATA_TAXID = 6843
MYRIAPODA_TAXID = 61985


def add_arthropod_group_attribute(tree, ncbi):
    """
    Add leaf.arthropod_group to each leaf.

    Hexapoda leaves get:
        leaf.arthropod_group = "Hexapoda"

    Chelicerata leaves get:
        leaf.arthropod_group = "Chelicerata"

    Myriapoda leaves get:
        leaf.arthropod_group = "Myriapoda"

    Other leaves get their own original name, so they are retained separately.
    """
    for leaf in tree.iter_leaves():
        old_name = leaf.name
        taxid = resolve_taxid(ncbi, old_name)

        if taxid is None:
            try:
                taxid=resolve_taxid(ncbi, old_name.strip().split(" ")[0])
            except:
                    
                sys.stderr.write(
                    f"WARNING: Could not resolve {old_name}; keeping original label\n"
                )
                leaf.order=old_name
                continue

        lineage = get_lineage(ncbi, taxid)

        if lineage is None:
            sys.stderr.write(
                f"WARNING: Could not get lineage for {leaf.name}; keeping as separate leaf\n"
            )
            leaf.add_feature("arthropod_group", leaf.name)
            continue

        if HEXAPODA_TAXID in lineage:
            leaf.order= "Hexapoda"
        elif CHELICERATA_TAXID in lineage:
            leaf.order="Chelicerata"
        elif MYRIAPODA_TAXID in lineage:
            leaf.order='Myriapoda'


In [180]:
add_arthropod_group_attribute(bernot,ncbi)
collapse_monophyletic_terminal_groups(bernot)

In [210]:
## keep myriapoda and chelicerata and hexapoda as backbone elements
keep=set(df[~df['class'].isin(['Arachnida','Pycnogonida','Merostomata','Diplopoda','Chilopoda','Insecta','Collembola'])]['order'])|set(['Myriapoda','Hexapoda','Chelicerata'])

In [211]:
## keep Notostraca, top hit in NR for 2 htgs but not in GenBank annotations
keep=keep|set(['Notostraca'])

In [183]:
bernot.prune(keep,preserve_branch_length=True)

In [184]:
print(bernot)


                  /-Pollicipedomorpha
               /-|
              |   \-Balanomorpha
              |
            /-|      /-Stomatopoda
           |  |   /-|
           |  |  |  |   /-Decapoda
           |  |  |   \-|
           |   \-|      \-Euphausiacea
           |     |
           |     |   /-Amphipoda
           |      \-|
           |         \-Isopoda
         /-|
        |  |         /-Anostraca
        |  |      /-|
        |  |     |  |   /-Notostraca
        |  |     |   \-|
        |  |   /-|      \-Diplostraca
        |  |  |  |
      /-|  |  |  |      /-Siphonostomatoida
     |  |  |  |  |   /-|
     |  |   \-|   \-|   \-Harpacticoida
     |  |     |     |
     |  |     |      \-Calanoida
   /-|  |     |
  |  |  |      \-Hexapoda
  |  |  |
--|  |   \-Podocopida
  |  |
  |   \-Myriapoda
  |
   \-Chelicerata


In [185]:
bernot.write(
    outfile="datasets/phylogenetic_trees/Bernot_2023_pruned.nwk",
    format=1
)


## Fernandez myriapoda


In [16]:
fernandez = Tree("datasets/phylogenetic_trees/timetree_Fernandez_2015.nwk")
relabel_leaves_by_order(fernandez, ncbi)
collapse_monophyletic_terminal_groups(fernandez)


In [17]:
fernandez.prune(list(set(df[df['class'].isin(['Diplopoda','Chilopoda'])]['order'])),preserve_branch_length=True)

In [18]:
print(fernandez)


      /-Scutigeromorpha
   /-|
  |   \-Scolopendromorpha
--|
  |   /-Polydesmida
   \-|
      \-Spirostreptida


In [30]:
fernandez.write(
    outfile="datasets/phylogenetic_trees/timetree_Fernandez_2015_pruned.nwk",
    format=1
)


## Howard Chelicerata

In [22]:
howard = Tree("datasets/phylogenetic_trees/timetree_Howard_2020.nwk")

relabel_leaves_by_order(howard, ncbi)
collapse_monophyletic_terminal_groups(howard)


In [23]:
print(howard)


               /-Hanseniella
            /-|
           |   \-Glomerida
         /-|
        |  |   /-Scolopendromorpha
        |   \-|
        |      \-Scutigeromorpha
        |
        |      /-Decapoda
      /-|     |
     |  |   /-|   /-Diplostraca
     |  |  |  |  |
     |  |  |  |  |         /-Agaricales
     |  |  |   \-|      /-|
     |  |  |     |   /-|   \-Poales
     |   \-|     |  |  |
     |     |      \-|   \-Blattodea
     |     |        |
     |     |         \-Entomobryomorpha
     |     |
     |      \-Arguloida
     |
     |               /-Mesostigmata
     |            /-|
     |           |   \-Ixodida
     |           |
     |           |            /-Sarcoptiformes
     |           |           |
     |           |         /-|      /-Malvales
     |         /-|        |  |   /-|
     |        |  |        |   \-|   \-Sarcoptiformes
   /-|        |  |      /-|     |
  |  |        |  |     |  |      \-Sarcoptiformes
  |  |        |  |   /-|  |
  |  |        |  |  |

In [24]:
G = howard.search_nodes(name="Malvales")[0]
G.detach()
print(howard)


               /-Hanseniella
            /-|
           |   \-Glomerida
         /-|
        |  |   /-Scolopendromorpha
        |   \-|
        |      \-Scutigeromorpha
        |
        |      /-Decapoda
      /-|     |
     |  |   /-|   /-Diplostraca
     |  |  |  |  |
     |  |  |  |  |         /-Agaricales
     |  |  |   \-|      /-|
     |  |  |     |   /-|   \-Poales
     |   \-|     |  |  |
     |     |      \-|   \-Blattodea
     |     |        |
     |     |         \-Entomobryomorpha
     |     |
     |      \-Arguloida
     |
     |               /-Mesostigmata
     |            /-|
     |           |   \-Ixodida
     |           |
     |           |            /-Sarcoptiformes
     |           |         /-|
     |         /-|        |  |   /- /-Sarcoptiformes
     |        |  |      /-|   \-|
   /-|        |  |     |  |      \-Sarcoptiformes
  |  |        |  |   /-|  |
  |  |        |  |  |  |   \-Sarcoptiformes
  |  |        |   \-|  |
  |  |        |     |   \-Sarcoptifo

In [25]:
collapse_monophyletic_terminal_groups(howard)

In [26]:
howard.prune(list(set(df[df['class'].isin(['Arachnida','Pycnogonida','Merostomata'])]['order'])),preserve_branch_length=True)

In [27]:
print(howard)


               /-Mesostigmata
            /-|
           |   \-Ixodida
         /-|
        |  |   /-Trombidiformes
        |   \-|
      /-|      \-Sarcoptiformes
     |  |
     |  |      /-Pseudoscorpiones
     |  |   /-|
   /-|   \-|   \-Scorpiones
  |  |     |
  |  |      \-Araneae
--|  |
  |   \-Xiphosura
  |
   \-Pantopoda


In [31]:
howard.write(
    outfile="datasets/phylogenetic_trees/timetree_Howard_2020_pruned.nwk",
    format=1
)

## starting with bernot backbone, replace chelicerata, myriapoda, hexapoda trees

In [250]:
## bernot
a=Tree("datasets/phylogenetic_trees/Bernot_2023_pruned.nwk").write(format=1)

In [251]:
hexapod=Tree("datasets/phylogenetic_trees/timetree_Misof_2014_insect_orders_collapsed_pruned.nwk")
farthest_leaf, distance = hexapod.get_farthest_leaf()
d=571.559-distance
s=hexapod.write(format=1).replace(";",f":{d}")
a=a.replace("Hexapoda:571.559",s)

In [252]:
hexapod=Tree("datasets/phylogenetic_trees/timetree_Howard_2020_pruned.nwk")
farthest_leaf, distance = hexapod.get_farthest_leaf()
d=640.285-distance
s=hexapod.write( format=1).replace(";",f":{d}")
a=a.replace("Chelicerata:640.285",s)

In [253]:
hexapod=Tree("datasets/phylogenetic_trees/timetree_Fernandez_2015_pruned.nwk")
farthest_leaf, distance = hexapod.get_farthest_leaf()
d=636.945-distance
s=hexapod.write( format=1).replace(";",f":{d}")
a=a.replace("Myriapoda:636.945",s)

In [202]:
from ete3 import NCBITaxa

ncbi = NCBITaxa()

def get_infraorder_taxid(taxid):
    lineage = ncbi.get_lineage(taxid)
    ranks = ncbi.get_rank(lineage)

    infraorder_taxid = next(
        (tid for tid in lineage if ranks.get(tid) == "infraorder"),
        None
        
    )
    if infraorder_taxid!=None:
        infraorder_name = ncbi.get_taxid_translator([ infraorder_taxid ])[ infraorder_taxid ]
        return infraorder_name

In [203]:
for index, row in df.iterrows():
    tid=row['Organism Taxonomic ID']
    df.loc[index,['infraorder']]=get_infraorder_taxid(tid)


In [204]:
def relabel_leaves_by_infraorder(tree, ncbi):
    """
    Rename each remaining hexapod leaf to its infraorder name.

    Assumes the tree has already been pruned to insects only.
    """
    for leaf in tree.iter_leaves():
        old_name = leaf.name
        taxid = resolve_taxid(ncbi, old_name)

        if taxid is None:
            try:
                taxid=resolve_taxid(ncbi, old_name.strip().split(" ")[0])
            except:
                    
                sys.stderr.write(
                    f"WARNING: Could not resolve {old_name}; keeping original label\n"
                )
                leaf.infraorder=old_name
                continue
       

        infraorder_name = get_rank_name_from_taxid(ncbi, taxid, "infraorder")

        if infraorder_name is None:
            sys.stderr.write(
                f"WARNING: Could not find infraorder for {old_name}; keeping original label\n"
            )
            leaf.order=old_name
            continue
 
 
        leaf.order = infraorder_name

In [254]:
d=Tree('datasets/phylogenetic_trees/Wiegmann_2011_treemble_figS3.nwk')
keep=list(df[df.order=='Diptera']['family'])
d.prune(keep,preserve_branch_length=True)


In [255]:
d=d.write(format=1)
d=Tree(d)

In [256]:
print(a)

((((((Pollicipedomorpha:158.347,Balanomorpha:158.347):414.657,((Stomatopoda:465.986,(Decapoda:422.466,Euphausiacea:422.466):43.5201):29.4215,(Amphipoda:465.176,Isopoda:465.176):30.2315):77.5958):29.1291,(((Anostraca:450.796,(Notostraca:364.264,Diplostraca:364.264):86.5321):86.8004,((Siphonostomatoida:314.084,Harpacticoida:314.084):93.8867,Calanoida:407.97):129.626):33.9626,((((((((((Lepidoptera:207.201,Trichoptera:207.201):82.4488,(Diptera:242.743,Siphonaptera:242.743):46.9067):37.0421,(Coleoptera:307.709,Neuroptera:307.709):18.983):17.9882,Hymenoptera:344.68):16.8525,Psocodea:361.532):11.7545,(Hemiptera:339.128,Thysanoptera:339.128):34.1591):13.6024,((Blattodea:231.271,Phasmatodea:231.271):16.5734,Orthoptera:247.845):139.045):19.5621,(Ephemeroptera:362.454,Odonata:362.454):43.9968):14.0978,Zygentoma:420.549):58.5756,(Symphypleona:242.695,Entomobryomorpha:242.695):236.43):92.43349999999998):30.5737):13.6315,Podocopida:615.764):21.1807,((Scutigeromorpha:442.691,Scolopendromorpha:442.691

In [257]:
farthest_leaf, distance = d.get_farthest_leaf()
di=242.743-distance
s=d.write(format=1).replace(";",f":{di}")
a=a.replace("Diptera:242.743",s)
print(Tree(a))
Tree(a).write(
    outfile="outputs/final_concatenated_arthropod_order.nwk",
    format=1
)


                  /-Pollicipedomorpha
               /-|
              |   \-Balanomorpha
              |
            /-|      /-Stomatopoda
           |  |   /-|
           |  |  |  |   /-Decapoda
           |  |  |   \-|
           |   \-|      \-Euphausiacea
           |     |
           |     |   /-Amphipoda
           |      \-|
           |         \-Isopoda
           |
           |         /-Anostraca
           |      /-|
           |     |  |   /-Notostraca
           |     |   \-|
           |   /-|      \-Diplostraca
           |  |  |
           |  |  |      /-Siphonostomatoida
           |  |  |   /-|
           |  |   \-|   \-Harpacticoida
           |  |     |
           |  |      \-Calanoida
           |  |
           |  |                              /-Lepidoptera
           |  |                           /-|
         /-|  |                          |   \-Trichoptera
        |  |  |                          |
        |  |  |                          |               /

In [258]:
a=Tree(a)

In [259]:

keep2=set([x.name for x in a if x.name not in keep])|set(['Culicidae','Drosophilidae','Stratiomyidae', 'Sciaridae','Psychodidae','Cecidomyiidae'])
a.prune(keep2,preserve_branch_length=True)
for x in a:
    if x.name=='Culicidae':
        x.name='Culicomorpha'
    if x.name=='Drosophilidae':
        x.name='Muscomorpha'
print(a)


                  /-Pollicipedomorpha
               /-|
              |   \-Balanomorpha
              |
            /-|      /-Stomatopoda
           |  |   /-|
           |  |  |  |   /-Decapoda
           |  |  |   \-|
           |   \-|      \-Euphausiacea
           |     |
           |     |   /-Amphipoda
           |      \-|
           |         \-Isopoda
           |
           |         /-Anostraca
           |      /-|
           |     |  |   /-Notostraca
           |     |   \-|
           |   /-|      \-Diplostraca
           |  |  |
           |  |  |      /-Siphonostomatoida
           |  |  |   /-|
         /-|  |   \-|   \-Harpacticoida
        |  |  |     |
        |  |  |      \-Calanoida
        |  |  |
        |  |  |                              /-Lepidoptera
        |  |  |                           /-|
        |  |  |                          |   \-Trichoptera
        |  |  |                          |
        |  |  |                          |         /-Psych

In [260]:
a.write(
    outfile="outputs/final_concatenated_arthropod_order_collapsed_diptera.nwk",
    format=1
)

In [261]:
t=set([x.name for x in a])
for index, row in df.iterrows():
    if row.order not in t and row.infraorder not in t and row.family not in t:
        print(row.order)

In [159]:
len(t)

49

In [289]:
!cp ../Bradysia_coprophila/itol_arthropod_donor_annotation_file.txt ./outputs/

In [283]:
a=Tree("outputs/final_concatenated_arthropod_order_collapsed_diptera.nwk",format=1)

In [284]:
import pandas as pd
dfriso=pd.read_csv("outputs/per_protein_diamond_data.tsv",sep="\t",index_col=0)
htgdf=dfriso[dfriso.tree_HTG]
from collections import Counter
cdict=Counter(htgdf[htgdf.phylum=='Arthropoda']['order'])
for x in cdict.keys():
    n=cdict[x]
    p=round(100*n/htgdf.shape[0],2)
    x=a.search_nodes(name=x)[0]
    x.name=x.name+f"_N{n}p={p}"
print(a)


                  /-Pollicipedomorpha
               /-|
              |   \-Balanomorpha
              |
            /-|      /-Stomatopoda
           |  |   /-|
           |  |  |  |   /-Decapoda_N2p=0.13
           |  |  |   \-|
           |   \-|      \-Euphausiacea
           |     |
           |     |   /-Amphipoda
           |      \-|
           |         \-Isopoda
           |
           |         /-Anostraca_N1p=0.07
           |      /-|
           |     |  |   /-Notostraca_N2p=0.13
           |     |   \-|
           |   /-|      \-Diplostraca_N3p=0.2
           |  |  |
           |  |  |      /-Siphonostomatoida
           |  |  |   /-|
         /-|  |   \-|   \-Harpacticoida
        |  |  |     |
        |  |  |      \-Calanoida
        |  |  |
        |  |  |                              /-Lepidoptera_N12p=0.79
        |  |  |                           /-|
        |  |  |                          |   \-Trichoptera
        |  |  |                          |
        |  | 

In [285]:
a.write(
    outfile="outputs/arthropod_tree_for_paper.nwk",quoted_node_names='True',
    format=1
)

In [291]:

import numpy as np

f=open("outputs/itol_arthropod_donor_annotation_file.txt","a")
for x in a:
    x=x.name
    if x.split("_")[0] in cdict:
        size=10*np.log10(cdict[x.split("_")[0]])
        f.write(f"{x},2,{size},#000000,1,-1\n")
f.close()

In [292]:
cdict['Odonata']

0